#Transform Sprints Data
1. Read bronze sprints table
2. Keep only the columns required for analytics (Drop url column)
3. Standardize column names using snake_case (driverId → driver_id, constructorId → constructor_id,raceName → race_name,positiontext → finish_position_text )
4.Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position 
5. Filter out rows where season, round, constructor_id and driver_id is null (business key validation)
6. Remove duplicate records
7. Transform column values of race_name to Title Case
8. Write the transformed data to silver results table

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"


Step 1,2,3,4-Read bronze sprints table,Select only required columns and standardize the column names

In [0]:
sprints_df = (
    
    spark.table(bronze_table)
         .drop("url")
         .withColumnsRenamed({
                           "driverId":"driver_id",
                           "constructorId":"constructor_id",
                           "positionText":"finsih_position_text",
                           "raceName":"race_name",
                           "date": "race_date",
                           "grid":"grid_position",
                           "laps":"completed_laps",
                           "number":"car_number",
                           "position":"finish_position"})              
              )
display(sprints_df)


Step 5,6- Apply Data Quality checks
- Filter out rows where season, round, constructor_id and driver_id is null (business key validation)
- Remove duplicate records

In [0]:
from pyspark.sql import functions as F 
sprints_valid_df =(sprints_df
                   .filter(
                            F.col("season").isNotNull() & 
                            F.col("round").isNotNull() & 
                            F.col("constructor_id").isNotNull() & 
                            F.col("driver_id").isNotNull()
                                                            )
                   .dropDuplicates(["driver_id","season","round","constructor_id"]) 
                   )
sprints_df.count()-sprints_valid_df.count()

Step 7- Transform column values of race_name to Title Case

In [0]:
sprints_final_df = (sprints_valid_df
                            .withColumn("race_name",F.initcap(F.col("race_name")))

)



Step 8- Write the transformed data to silver sprints table

In [0]:
(
    sprints_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))